<a href="https://colab.research.google.com/github/Euriks27/macrolab-site/blob/main/C%C3%B3pia_de_Untitled35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy sqlalchemy python-bcb sidrapy pandas-datareader yfinance statsmodels scikit-learn

In [ ]:
# ==============================================================================
# MacroLab – Framework de Credibilidade Fiscal e Monetária (Otimizado)
# ==============================================================================

!pip install python-bcb # Added back to ensure module availability
!pip install sidrapy # Added to ensure sidrapy is installed

import pandas as pd
import numpy as np
import datetime
from sqlalchemy import create_engine
from bcb import sgs, Expectativas
import sidrapy
import pandas_datareader.data as web
import yfinance as yf
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller # Corrected import path
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from sklearn.preprocessing import StandardScaler

class MacroLab:
    def __init__(self, db_path='sqlite:///macrolab_core.db'):
        self.engine = create_engine(db_path)
        self.df = None
        self.results = {}
        print("🚀 MacroLab – Data Hub Ativo.")

    def collect_and_store_data(self, start='2010-01-01', end=datetime.date.today().strftime('%Y-%m-%d')):
        """Coleta dados de diversas fontes e os armazena no Data Hub."""
        print("🌍 Coletando dados de diversas fontes e armazenando no Data Hub...")
        try:
            # 1. Coleta de dados do BCB (SGS)
            # BCB API has a ~10 year limit for daily series. Adjusting start date for BCB.
            end_dt = datetime.datetime.strptime(end, '%Y-%m-%d').date()
            bcb_start_dt = end_dt - datetime.timedelta(days=10*365) # Approximately 10 years
            bcb_start = bcb_start_dt.strftime('%Y-%m-%d')

            bcb_series_codes = {'12': 'CDI', '4390': 'SELIC'}
            df_bcb = sgs.get(list(bcb_series_codes.keys()), start=bcb_start, end=end).rename(columns=bcb_series_codes)

            # 2. Coleta de dados do Yahoo Finance (original start date can be used)
            df_yfinance_raw = yf.download('^BVSP', start=start, end=end)
            df_yfinance = pd.DataFrame() # Initialize as empty
            # Changed to use 'Close' column as 'Adj Close' is often missing for ^BVSP
            # Now explicitly handling MultiIndex columns from yfinance
            if not df_yfinance_raw.empty and ('Close', '^BVSP') in df_yfinance_raw.columns:
                df_yfinance = df_yfinance_raw[('Close', '^BVSP')].to_frame(name='BVSP')
            else:
                print(f"⚠️ Não foi possível obter dados de '^BVSP' do Yahoo Finance. DataFrame vazio ou 'Close' ausente. Pulando.")

            # Unindo e padronizando os dados
            # Concatenar apenas se os DataFrames não estiverem vazios
            dataframes_to_concat = []
            if not df_bcb.empty:
                dataframes_to_concat.append(df_bcb)
            if not df_yfinance.empty:
                dataframes_to_concat.append(df_yfinance)

            if not dataframes_to_concat:
                print("❌ Nenhuma fonte de dados retornou dados válidos para armazenamento.")
                return # Exit if no data to store

            df_combined = pd.concat(dataframes_to_concat, axis=1)
            df_combined = df_combined.ffill().dropna()

            if df_combined.empty:
                print("❌ DataFrame combinado está vazio após pré-processamento. Nenhuma observação para armazenar.")
                return # Exit if combined data is empty

            # Transformando para o formato 'observacoes'
            df_temp = df_combined.reset_index()
            # Identify the column name that holds the date after reset_index (usually 'Date' or 'index')
            date_col_name = df_temp.columns[0]

            df_melted = df_temp.melt(id_vars=[date_col_name], var_name='id_serie', value_name='valor')
            df_melted.rename(columns={date_col_name: 'data_referencia'}, inplace=True)
            df_melted['data_referencia'] = pd.to_datetime(df_melted['data_referencia']).dt.date # Armazenar como data simples

            # Armazenando no banco de dados
            df_melted.to_sql('observacoes', self.engine, if_exists='replace', index=False)
            print("✅ Dados coletados e armazenados com sucesso no Data Hub.")
        except Exception as e:
            print(f"❌ Erro ao coletar e armazenar dados: {e}")

    def load_data(self, start='2010-01-01'):
        """Carrega dados do SQL Hub. Se o banco estiver vazio, coleta via API."""
        print("📥 Carregando dados do Data Hub...")
        try:
            # Consulta otimizada
            query = "SELECT data_referencia, id_serie, valor FROM observacoes WHERE data_referencia >= ?"
            df_sql = pd.read_sql(query, self.engine, params=(start,))

            if df_sql.empty:
                print("⚠️ Tabela 'observacoes' vazia ou não encontrada. Por favor, execute a rotina de coleta primeiro.")
                self.df = None # Ensure df is None if nothing is loaded
                return

            self.df = df_sql.pivot(index='data_referencia', columns='id_serie', values='valor')
            self.df.index = pd.to_datetime(self.df.index) # Convert index to datetime for time series operations
            self.df = self.df.ffill().dropna()
            print("✅ Dados carregados e alinhados com sucesso.")
        except Exception as e:
            print(f"⚠️ Erro no hub: {e}. Execute a rotina de coleta (Coletor).")
            self.df = None # Ensure df is None if loading fails

    def adf_tests(self):
        """Teste de raiz unitária para rigor acadêmico."""
        print("\n📊 Teste ADF (Raiz Unitária):")
        for col in self.df.columns:
            p = adfuller(self.df[col])[1]
            print(f"{col}: p-valor = {p:.4f}")

    def run_var_model(self, lags=12):
        """Modelo VAR com foco em dinâmica de curto prazo."""
        model = VAR(self.df)
        self.results['var'] = model.fit(lags)
        print("\n📈 Modelo VAR estimado.")
        print(self.results['var'].summary())

    def compute_fsi(self):
        """Cálculo do Índice de Estresse Fiscal (FSI)."""
        scaler = StandardScaler()
        # Ensure self.df contains numeric data suitable for scaling
        numeric_df = self.df.select_dtypes(include=[np.number])
        if not numeric_df.empty:
            self.df['FSI'] = scaler.fit_transform(numeric_df).mean(axis=1)
            print("ðŸ“ˆ FSI calculado e integrado.")
        else:
            print("⚠️ Não há dados numéricos para calcular o FSI.")

    def calculate_correlation(self):
        """Calcula e exibe a matriz de correlação das séries no DataFrame."""
        print("\nCorrelation Matrix:")
        if self.df is not None and not self.df.empty:
            correlation_matrix = self.df.corr()
            display(correlation_matrix)
        else:
            print("⚠️ DataFrame is empty, cannot calculate correlation.")

    def check_data_integrity(self):
        """Verifica se existem valores nulos ou atípicos que possam enviesar a tese."""
        print("\n🔍 Relatório de Integridade de Dados:")
        if self.df is not None and not self.df.empty:
            missing_data = self.df.isnull().sum()
            print(missing_data[missing_data > 0])

            # Se houver muitos dados faltantes, talvez seja necessário um alerta para a banca
            if missing_data.sum() > 0:
                print("⚠️ Atenção: Lacunas identificadas. Recomenda-se interpolação linear.")
            else:
                print("✅ Nenhum dado ausente encontrado.")
        else:
            print("⚠️ DataFrame está vazio, não é possível verificar a integridade dos dados.")

# --- EXECUÇÃO (Para copiar e colar no Colab) ---
if __name__ == "__main__":
    # Inicialização
    lab = MacroLab()

    # 1. Coletar e armazenar dados para garantir que a base esteja populada
    lab.collect_and_store_data()

    # 2. Carregar dados do banco de dados
    lab.load_data()

    # 3. Execução do fluxo de trabalho apenas se os dados foram carregados com sucesso
    if lab.df is not None and not lab.df.empty:
        lab.adf_tests()
        lab.compute_fsi()
        lab.calculate_correlation()
        lab.check_data_integrity() # Call the new method

        # Opcional: Modelagem
        # lab.run_var_model()
    else:
        print("❌ Não foi possível carregar dados para análise. Verifique a rotina de coleta e carregamento.")

🚀 MacroLab – Data Hub Ativo.
🌍 Coletando dados de diversas fontes e armazenando no Data Hub...
❌ Erro ao coletar e armazenar dados: SGS time series code=12 server error (status 502)
📥 Carregando dados do Data Hub...
✅ Dados carregados e alinhados com sucesso.

📊 Teste ADF (Raiz Unitária):
BVSP: p-valor = 0.7336
CDI: p-valor = 0.8683
SELIC: p-valor = 0.6924
ðŸ“ˆ FSI calculado e integrado.

Correlation Matrix:


id_serie,BVSP,CDI,SELIC,FSI
id_serie,,,,
BVSP,1.000000,0.287447,0.268448,0.630900
CDI,0.287447,1.000000,0.985062,0.921479
SELIC,0.268448,0.985062,1.000000,0.913775
FSI,0.630900,0.921479,0.913775,1.000000



🔍 Relatório de Integridade de Dados:
Series([], dtype: int64)
✅ Nenhum dado ausente encontrado.


In [ ]:
print("\n--- Debugging Yahoo Finance Data Collection ---")
try:
    # Define the same parameters as in the MacroLab method
    start = '2010-01-01'
    end = datetime.date.today().strftime('%Y-%m-%d')

    # Download the data
    debug_df_yfinance_raw = yf.download('^BVSP', start=start, end=end)

    print(f"Raw Yahoo Finance DataFrame shape: {debug_df_yfinance_raw.shape}")
    print(f"Raw Yahoo Finance DataFrame columns: {debug_df_yfinance_raw.columns.tolist()}")
    print("Raw Yahoo Finance DataFrame head:")
    display(debug_df_yfinance_raw.head())

    if debug_df_yfinance_raw.empty:
        print("Conclusion: DataFrame is empty.")
    elif 'Adj Close' not in debug_df_yfinance_raw.columns:
        print("Conclusion: 'Adj Close' column is missing.")
    else:
        print("Conclusion: 'Adj Close' column is present and DataFrame is not empty.")

except Exception as e:
    print(f"An error occurred during Yahoo Finance debug download: {e}")
print("----------------------------------------------")

/tmp/ipykernel_7837/3971455560.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  debug_df_yfinance_raw = yf.download('^BVSP', start=start, end=end)
[*********************100%***********************]  1 of 1 completed


--- Debugging Yahoo Finance Data Collection ---
Raw Yahoo Finance DataFrame shape: (4086, 5)
Raw Yahoo Finance DataFrame columns: [('Close', '^BVSP'), ('High', '^BVSP'), ('Low', '^BVSP'), ('Open', '^BVSP'), ('Volume', '^BVSP')]
Raw Yahoo Finance DataFrame head:


Price,Close,High,Low,Open,Volume
Ticker,^BVSP,^BVSP,^BVSP,^BVSP,^BVSP
Date,,,,,
2010-01-04,70045.0,70081.0,68587.0,68587.0,1655400
2010-01-05,70240.0,70595.0,69928.0,70046.0,1984200
2010-01-06,70729.0,70937.0,70016.0,70237.0,2243600
2010-01-07,70451.0,70723.0,70045.0,70723.0,1555000
2010-01-08,70263.0,70766.0,70158.0,70455.0,1634400


Conclusion: 'Adj Close' column is missing.
----------------------------------------------
